In [72]:
# библиотеки
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import scipy.stats as sts
from itertools import combinations

In [73]:
# load data
%store -r data

In [78]:
data = data[(data['start_date'].dt.year == 2020)]
print(data['start_date'].dt.year.value_counts())

start_date
2020    403
Name: count, dtype: int64


In [79]:
# Выделяем количественные переменные
quantitative_data = data[['score', 'scored_by', 'volumes', 'chapters']]

# Строим корреляционную матрицу
correlation_matrix = quantitative_data.corr().round(3)
print(correlation_matrix)

           score  scored_by  volumes  chapters
score      1.000      0.122    0.006     0.025
scored_by  0.122      1.000    0.374     0.383
volumes    0.006      0.374    1.000     0.839
chapters   0.025      0.383    0.839     1.000


### Связь целевой переменной score с факторами

Все связи с score крайне слабые:

scored_by: 0.122 → минимальная положительная связь

volumes: 0.006 → практически отсутствует

chapters: 0.025 → практически отсутствует

Вывод: Ни один из рассматриваемых факторов не оказывает значимого влияния на рейтинг манги.

### Межфакторные корреляции

Наблюдается две значимые взаимосвязи:

volumes ↔ chapters: 0.839 → очень сильная зависимость

Ожидаемо, так как количество глав напрямую связано с количеством томов

scored_by ↔ volumes: 0.374 и scored_by ↔    chapters: 0.383 → умеренные связи

Более длинные манги получают несколько больше оценок

In [80]:
variables = ['score', 'scored_by', 'volumes', 'chapters']
for (var1, var2) in combinations(variables, 2):
    r, p = sts.pearsonr(data[var1], data[var2])
    print(f"{var1} & {var2}: r = {r:.3f}, p-value = {p:.3f}")

score & scored_by: r = 0.122, p-value = 0.015
score & volumes: r = 0.006, p-value = 0.906
score & chapters: r = 0.025, p-value = 0.610
scored_by & volumes: r = 0.374, p-value = 0.000
scored_by & chapters: r = 0.383, p-value = 0.000
volumes & chapters: r = 0.839, p-value = 0.000


Для целевой переменной score:

```scored_by (r=0.122, p=0.015):```

Связь слабая, но статистически значимая.

На 95% уровне уверенности можно утверждать, что связь между количеством оценок и рейтингом существует, хотя и минимальная.

```volumes (r=0.006, p=0.906) и chapters (r=0.025, p=0.610):```

Корреляции незначимы (p > 0.05).

Нет оснований отвергать нулевую гипотезу (связи с рейтингом нет).

Межфакторные корреляции:

Все связи между предикторами (scored_by, volumes, chapters) значимы (p < 0.001).

Особенно сильная зависимость между volumes и chapters (r=0.839), что указывает на мультиколлинеарность.

In [83]:
# Преобразуем статус в бинарную переменную
data['status_binary'] = data['status'].apply(lambda x: 1 if x == 'finished' else 0)
print(data['status_binary'].value_counts())

if len(data['status_binary'].unique()) > 1:
    r_pb, p_value = sts.pointbiserialr(data['status_binary'], data['score'])
    print(f"Коэффициент корреляции: {r_pb:.3f}, p-value: {p_value:.3f}")
else:
    print("Ошибка: статус не варьируется (все значения одинаковы).")

status_binary
1    399
0      4
Name: count, dtype: int64
Коэффициент корреляции: -0.018, p-value: 0.724


In [84]:
# Создадим бинарную переменную для жанра "Romance"
data['is_romance'] = data['genres'].apply(lambda x: 1 if 'Romance' in x else 0)
print(data['is_romance'].value_counts())

# Проверка корреляции
if len(data['is_romance'].unique()) > 1:
    r_pb, p_value = sts.pointbiserialr(data['is_romance'], data['score'])
    print(f"Коэффициент корреляции: {r_pb:.3f}, p-value: {p_value:.3f}")
else:
    print("Ошибка: статус не варьируется (все значения одинаковы).")

is_romance
0    296
1    107
Name: count, dtype: int64
Коэффициент корреляции: 0.043, p-value: 0.389
